# VECTOR TILING FOR MAP GENERALIZATION

Generalization means creating many copies of a single ‘source of truth’ and tailoring them to the view context

Doing this on the fly is labor intensive in a static paper mapping context, especially en masse with variable map scales and contexts (urban, semi-urban, rural) 

Value in adopting dynamic mapping techniques into the existing processing workflow:

- Vector tiling: highly optimized, customizable, retains records’ attributes (versus image tiles)
- Customization, filtering, alignment between 'in-house' GRID3 data and third-party sources like OpenStreetMap
- API endpoint targets: maps can diff and grow as the sources of truth do
- TIPPECANOE: Incredible and fast processing library, somewhat unpredictable w/ emergent effects when processing geometries in a ‘recursive tiled’ manner, but benefits outweigh headaches
- Plus: Self-hosting interest (docker/containerization): anxiety after ‘open’ data portals going offline nationwide



<!-- - What works?

- What doesn't? -->
<!-- 
Note: “Emergent properties” -> usually helpful automation, as long as there are constraints, but need to watch for weirdness and misalignment with what a legend would otherwise say a layer/category should look like

(For instance, interpolation between color steps, etc) -->

===

# Processing pipeline

1. **Download** - Fetch Overture Maps and GRID3 (AGOL feature services) data for specified extent (as GeoParquet file)
2. **Convert to FlatGeobuf** - Transform GeoParquet to FlatGeobuf for compatibility + efficiency
3. **Tile** - Generate PMTiles using tippecanoe with bespoke settings per-layer
4. **View** - using maplibre open spec

## File formats
- **GeoParquet (.parquet)** - Download format (compact, fast queries via "duckquery")
- **FlatGeobuf (.fgb)** - Convert for optimal tippecanoe library support
- **GeoJSON (.geojson)** - Fallback support for small datasets
- **pmTiles** - Dynamic vector tiles, served from single static file

## CONFIG

.env -> config.py imports environment vars

In [1]:
# ============================================================
# CONFIGURATION - Run this cell first
# ============================================================
# This cell initializes all configuration and should be run 
# first. Re-run this cell to reload configuration changes.
# ============================================================

import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Setup paths
notebook_dir = Path.cwd()
print(f"Current notebook directory: {notebook_dir}")
processing_dir = notebook_dir  # preprocessing
repo_root = notebook_dir     # basemap (repository root)

# Add processing directory to path
if str(processing_dir) not in sys.path:
    sys.path.insert(0, str(processing_dir))

# Load environment variables from REPOSITORY ROOT (monorepo-wide .env)
env_path = repo_root / '.env'
load_dotenv(env_path)
print(f"✓ Loaded environment from repository root: {env_path}")
print(f"  DATA_DISK = {os.environ.get('DATA_DISK', 'not set')}")

# ============================================================
# FIX PROJ LIBRARY PATH FOR GDAL/OGR
# ============================================================
# Set PROJ_LIB to point to the micromamba environment's proj data
proj_lib_path = Path.home() / "micromamba/envs/geoprocessing/share/proj"
if proj_lib_path.exists():
    os.environ['PROJ_LIB'] = str(proj_lib_path)
    print(f"✓ Set PROJ_LIB: {proj_lib_path}")
else:
    # Try alternative locations
    conda_prefix = os.environ.get('CONDA_PREFIX', '')
    if conda_prefix:
        alt_proj = Path(conda_prefix) / "share/proj"
        if alt_proj.exists():
            os.environ['PROJ_LIB'] = str(alt_proj)
            print(f"✓ Set PROJ_LIB: {alt_proj}")
        else:
            print(f"⚠️  Warning: PROJ data not found at {proj_lib_path}")
            print(f"   GDAL may have issues with coordinate transformations")
    else:
        print(f"⚠️  Warning: PROJ data not found at {proj_lib_path}")

# Import configuration (will also load .env via config.py)
from config import (
    get_config,
    ensure_directories,
    print_config_summary,
    SCRIPTS_DIR,
    OUTPUT_DIR,
    OVERTURE_DATA_DIR,
    GRID3_DATA_DIR,
    SCRATCH_DIR,
    SCRATCH_BOUNDARIES_DIR,
    SCRATCH_POIS_DIR,
    SCRATCH_EXTENTS_DIR,
)

# Import processing functions
from scripts import (
    download_overture_data,
    convert_file,
    convert_parquet_to_fgb,
    batch_convert_directory,
    convert_geodata_to_fgb,
    batch_convert_geodata,
    process_to_tiles,
    create_tilejson,
    download_arcgis_data,
    batch_download_arcgis_layers,
)

# Additional libraries
import pandas as pd
import polars as pl
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# GLOBAL CONFIGURATION - Available in all cells below
# ============================================================
CONFIG = get_config()

# ============================================================
# EXTENT CONFIGURATION - SINGLE SOURCE OF TRUTH
# ============================================================
# Extent is now configured in .env file (repository root)
# To change the geographic area, edit .env and restart kernel
# 
# Current extent values from .env:
print(f"\n=== EXTENT FROM ENVIRONMENT (.env) ===")
print(f"  West (lon_min):  {os.environ.get('EXTENT_WEST', 'not set')}")
print(f"  South (lat_min): {os.environ.get('EXTENT_SOUTH', 'not set')}")
print(f"  East (lon_max):  {os.environ.get('EXTENT_EAST', 'not set')}")
print(f"  North (lat_max): {os.environ.get('EXTENT_NORTH', 'not set')}")
print(f"  Buffer (degrees): {os.environ.get('EXTENT_BUFFER', 'not set')}")
print(f"\n  Combined tuple: {CONFIG['extent']['coordinates']}")
print(f"  Buffer: {CONFIG['extent']['buffer_degrees']} degrees")

# DO NOT override extent here - edit .env instead!
# CONFIG["extent"]["coordinates"] is automatically loaded from .env

# ============================================================
# DISK SPACE MANAGEMENT
# ============================================================
# Automatically remove source files after successful conversion to save disk space
# FlatGeobuf (.fgb) files are kept as intermediary files for fast PMTiles regeneration
CONFIG["fgb_conversion"]["cleanup_source"] = True  # Set to False to keep source files

# Processing options (can still be customized here)
# tiling.input_dirs is set in config.py → [SCRATCH_BOUNDARIES_DIR, SCRATCH_POIS_DIR, SCRATCH_EXTENTS_DIR]
# Override here only if you need a different subset, e.g.:
# CONFIG["tiling"]["input_dirs"] = [str(SCRATCH_BOUNDARIES_DIR)]
CONFIG["download"]["verbose"] = True
CONFIG["conversion"]["verbose"] = True
CONFIG["tiling"]["verbose"] = True
CONFIG["tiling"]["parallel"] = True

# Create directories and verify
ensure_directories()

# Verification
print("\n=== CONFIGURATION VERIFICATION ===")
print(f"Repository root:       {repo_root}")
print(f"Environment .env:      {env_path}")
print(f"Environment DATA_DISK: {os.environ.get('DATA_DISK', 'NOT SET')}")
print(f"Config uses:           {CONFIG['paths']['data_dir'].parent}")

print_config_summary(CONFIG)

# Tiling input directories
print("\n=== TILING INPUT DIRECTORIES ===")
for d in CONFIG["tiling"]["input_dirs"]:
    d = Path(d)
    fgb_count = len(list(d.glob("*.fgb"))) if d.exists() else 0
    print(f"  {d.name}/  ({fgb_count} .fgb files)")

# Disk management info
print("\n=== DISK SPACE MANAGEMENT ===")
print(f"  Cleanup source files: {CONFIG['fgb_conversion']['cleanup_source']}")
if CONFIG['fgb_conversion']['cleanup_source']:
    print("  -> Source .parquet files will be removed after successful .fgb conversion")
    print("  -> FlatGeobuf (.fgb) files retained for fast PMTiles regeneration")
else:
    print("  -> Both .parquet and .fgb files will be kept")

print("\n✓ Configuration loaded - CONFIG available in all cells")
print("\n⚠️  To change extent: Edit .env file and restart kernel")
print("   All downloads, processing, and viewing will use the same extent")

Current notebook directory: /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing
✓ Loaded environment from repository root: /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing/.env
  DATA_DISK = /tmp/grid3_tiles
✓ Set PROJ_LIB: /home/mjh2241/micromamba/envs/geoprocessing/share/proj

=== EXTENT FROM ENVIRONMENT (.env) ===
  West (lon_min):  -33.8
  South (lat_min): -47.5
  East (lon_max):  71.8
  North (lat_max): 45.9
  Buffer (degrees): 0

  Combined tuple: (-33.8, -47.5, 71.8, 45.9)
  Buffer: 0.0 degrees

=== CONFIGURATION VERIFICATION ===
Repository root:       /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing
Environment .env:      /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing/.env
Environment DATA_DISK: /tmp/grid3_tiles
Config uses:           /tmp/grid3_tiles
PROJECT CONFIGURATION
Project root:        /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing
Scripts directory:   /home/mjh2241/GitHub/GRID3_mapTiles/preprocessing/scripts
Notebooks directory: /home/mjh2241/GitHub/GRID3_mapTile

## Download Overture Data with DuckDB 

Use the `downloadOverture.py` module to fetch geospatial data from Overture Maps

e.g., replace periodic geofabrik OSM ~shapefile~ fetches

In [ ]:
# Download Overture Maps data
print("=== STEP 1: DOWNLOADING OVERTURE DATA ===")
download_results = download_overture_data(
    extent=CONFIG["extent"]["coordinates"],
    buffer_degrees=CONFIG["extent"]["buffer_degrees"],
    template_path=str(CONFIG["paths"]["template_path"]),
    verbose=CONFIG["download"]["verbose"],
    project_root=str(CONFIG["paths"]["project_root"]),
    overture_data_dir=str(CONFIG["paths"]["overture_data_dir"])
)

print(f"Download completed: {download_results['success']}")
print(f"Sections processed: {download_results['processed_sections']}")
if download_results["errors"]:
    print(f"Errors encountered: {len(download_results['errors'])}")
    for error in download_results["errors"]:
        print(f"  - {error}")
print()

## Download ArcGIS Feature Server Data

Download geospatial data from hosted ArcGIS Feature Server REST API endpoints - can include any esri-hosted data as endpoint

- **Automatic pagination** - Handles ArcGIS's 1000-2000 feature limit per request
- **Spatial filtering** - Apply bounding box filter to download only features in aoi
- **formats** - Download as GeoJSON or directly convert to FlatGeobuf
- **Batch processing** - Download multiple layers with one function call

### GRID3 DRC Layers
- https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services

In [ ]:
# DIAGNOSTIC: Test ArcGIS Feature Server connections before downloading
print("=== DIAGNOSTIC: TESTING ARCGIS SERVICE CONNECTIONS ===\n")

from scripts import test_service_connection

# Test each service URL for accessibility and metadata
# NOTE: Services updated from v7_0 to v8_0 (January 2025)
test_urls = [
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0'
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_health_areas_v8_0/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_settlement_names_v8_0/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/COD_GRID3_health_facilities_v8_0/FeatureServer/0'
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_COD_Settlement_Extents_v3_1/FeatureServer/0',
    # 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0'
]

test_results = []
for url in test_urls:
    layer_name = url.split('/services/')[-1].split('/')[0]
    print(f"\nTesting: {layer_name}")
    print("-" * 70)
    
    result = test_service_connection(url, verbose=True)
    test_results.append({
        'layer': layer_name,
        'url': url,
        **result
    })

# Summary
print(f"\n{'='*70}")
print("CONNECTION TEST SUMMARY")
print(f"{'='*70}")
accessible_count = sum(1 for r in test_results if r['accessible'])
print(f"Total services tested: {len(test_results)}")
print(f"Accessible: {accessible_count}")
print(f"Failed: {len(test_results) - accessible_count}")

if accessible_count < len(test_results):
    print(f"\nFailed services:")
    for r in test_results:
        if not r['accessible']:
            print(f"  ✗ {r['layer']}")
            print(f"    Error: {r['error']}")
    
    print(f"\n⚠️  TROUBLESHOOTING TIPS:")
    print(f"  1. Check if you need to authenticate (VPN, organizational login)")
    print(f"  2. Verify the service URLs are still active")
    print(f"  3. Check if the organization changed permissions")
    print(f"  4. Try accessing the URL in a browser to see the error message")
else:
    print(f"\n✓ All services accessible - ready to download!")
    print(f"\n📝 SERVICE METADATA:")
    for r in test_results:
        if r['accessible']:
            print(f"\n  {r['layer']}:")
            if 'feature_count' in r:
                print(f"    Features: {r['feature_count']:,}")
            if 'geometry_type' in r:
                print(f"    Geometry: {r['geometry_type']}")
            if 'max_record_count' in r:
                print(f"    Max records/request: {r['max_record_count']:,}")

print(f"{'='*70}\n")


### ArcGIS Feature Server Downloads

**Finding Feature Server URLs:**
1. Browse your organization's ArcGIS REST Services Directory
2. Navigate to a specific layer (e.g., FeatureServer/0, FeatureServer/1)
3. Copy the full URL up to and including the layer number
4. The script will automatically append `/query` and handle parameters

**Spatial Filtering:**
- The `extent` parameter filters features to your bounding box (saves bandwidth & time)
- For global layers, omit `extent=None` to download all features
- Extent uses WGS84 coordinates: `(lon_min, lat_min, lon_max, lat_max)`

**Attribute Filtering:**
- Use `where` clause for SQL-based filtering: `'population > 10000'`
- Default `'1=1'` downloads all features

**Output Formats:**
- `"fgb"` (FlatGeobuf) - Recommended for direct tiling (streaming, indexed)
- `"geojson"` - more flexible, less optimal


- Large datasets (>100k features) automatically use pagination
- Downloads directly to scratch directory

In [ ]:
# Download ArcGIS Feature Server data (optional - skip if not needed)
print("=== STEP 2a: DOWNLOADING ARCGIS DATA  ===")
print(f"Current extent from CONFIG: {CONFIG['extent']['coordinates']}")
print(f"  Longitude: {CONFIG['extent']['coordinates'][0]} to {CONFIG['extent']['coordinates'][2]}")
print(f"  Latitude: {CONFIG['extent']['coordinates'][1]} to {CONFIG['extent']['coordinates'][3]}")

# ============================================================
# SPATIAL FILTERING OPTIONS
# ============================================================
# Option 1: NO SPATIAL FILTER (default - recommended for large extents)
download_extent = None
# Option 2: USE SPATIAL FILTER
# download_extent = CONFIG["extent"]["coordinates"]

print(f"\nSpatial filtering: {'DISABLED (downloading all features)' if download_extent is None else f'ENABLED {download_extent}'}")

# ── BOUNDARIES ──────────────────────────────────────────────────────────────
# All admin/operational boundary polygons → SCRATCH_BOUNDARIES_DIR
boundaries_layers = [
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_COD_health_zones_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_zones_v8_0',
    #     'where': '1=1'
    # },
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_health_areas_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_areas_v8_0',
    #     'where': '1=1'
    # },
]

if boundaries_layers:
    print(f"\n--- Downloading {len(boundaries_layers)} boundary layer(s) -> {SCRATCH_BOUNDARIES_DIR.name}/ ---")
    b_results = batch_download_arcgis_layers(
        layer_configs=boundaries_layers,
        output_dir=str(SCRATCH_BOUNDARIES_DIR),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in b_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

# ── SETTLEMENT EXTENTS ───────────────────────────────────────────────────────
# Settlement extent polygons → SCRATCH_EXTENTS_DIR
extents_layers = [
    {
        'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_NGA_settlement_extents_v4_0/FeatureServer/0',
        'name': 'GRID3_NGA_settlement_extents_v4_0',
        'where': '1=1'
    },
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/arcgis/rest/services/GRID3_COD_Settlement_Extents_v3_1/FeatureServer/0',
    #     'name': 'GRID3_COD_settlement_extents_v3_1',
    #     'where': '1=1'
    # },
]

if extents_layers:
    print(f"\n--- Downloading {len(extents_layers)} settlement extent layer(s) -> {SCRATCH_EXTENTS_DIR.name}/ ---")
    e_results = batch_download_arcgis_layers(
        layer_configs=extents_layers,
        output_dir=str(SCRATCH_EXTENTS_DIR),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in e_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

# ── POIs ─────────────────────────────────────────────────────────────────────
# Health facilities, settlement names, other point features → SCRATCH_POIS_DIR
poi_layers = [
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/COD_GRID3_health_facilities_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_health_facilities_v8_0',
    #     'where': '1=1'
    # },
    # {
    #     'url': 'https://services3.arcgis.com/BU6Aadhn6tbBEdyk/ArcGIS/rest/services/GRID3_COD_settlement_names_v8_0/FeatureServer/0',
    #     'name': 'GRID3_COD_settlement_names_v8_0',
    #     'where': '1=1'
    # },
]

if poi_layers:
    print(f"\n--- Downloading {len(poi_layers)} POI layer(s) -> {SCRATCH_POIS_DIR.name}/ ---")
    p_results = batch_download_arcgis_layers(
        layer_configs=poi_layers,
        output_dir=str(SCRATCH_POIS_DIR),
        extent=download_extent,
        output_format="fgb",
        verbose=CONFIG["download"]["verbose"],
        timeout=180
    )
    for layer in p_results['layers']:
        status = f"✓ {layer['name']}: {layer['feature_count']:,} features" if layer['success'] else f"✗ {layer['name']}: {layer.get('error', 'unknown error')}"
        print(f"  {status}")

if not any([boundaries_layers, extents_layers, poi_layers]):
    print("\nNo ArcGIS layers configured. Uncomment entries above to download data.")

#### note - spatial filtering is disabled by default when downloading from arcgis feature services. very large extents may trigger a 400 error. when in doubt, download the full extent and post-process (below) to an area of interest

In [ ]:
# OPTIONAL: Clip downloaded features to extent if needed

# import geopandas as gpd
# from pathlib import Path

# print("=== CLIPPING ARCGIS DATA TO EXTENT ===\n")

# # Get extent from CONFIG
# lon_min, lat_min, lon_max, lat_max = CONFIG["extent"]["coordinates"]
# print(f"Clipping extent: ({lon_min}, {lat_min}, {lon_max}, {lat_max})")

# # List of layers to clip
# layers_to_clip = ['settlement_extents']  # Add more as needed

# for layer_name in layers_to_clip:
#     input_file = CONFIG["paths"]["scratch_dir"] / f"{layer_name}.fgb"
    
#     if input_file.exists():
#         print(f"\nProcessing: {layer_name}")
        
#         # Read the file
#         gdf = gpd.read_file(input_file)
#         original_count = len(gdf)
#         print(f"  Original features: {original_count:,}")
        
#         # Clip to extent using GeoPandas spatial indexing
#         gdf_clipped = gdf.cx[lon_min:lon_max, lat_min:lat_max]
#         clipped_count = len(gdf_clipped)
#         print(f"  After clipping: {clipped_count:,}")
#         print(f"  Removed: {original_count - clipped_count:,} features outside extent")
        
#         # Overwrite the original file (or save with different name)
#         gdf_clipped.to_file(input_file, driver="FlatGeobuf")
#         print(f"  ✓ Updated: {input_file.name}")
#     else:
#         print(f"\n✗ {layer_name}.fgb not found")

# print("\n✓ Clipping complete!")


## Generate Centroids for Administrative Polygons

Generate interior centroid points for health zones and health areas. These will be used for single-label-per-polygon rendering in the map viewer (interior labels at lower zoom levels).

**Why centroids?**
- Guarantees one label per polygon (no duplicates across tile boundaries)
- `representative_point()` ensures label is always inside the polygon
- Preserves all attributes for label content
- Separate point layer is more efficient than point-based symbol placement on polygons

**NEW: Label Rotation for Best-Fit**
- Each centroid now includes a `label_rotation` attribute (in degrees)
- Calculated from the minimum rotated rectangle (oriented bounding box) of each polygon
- Labels rotate to align with the polygon's longest axis
- Ensures labels fit naturally within elongated or diagonal polygons
- Rotation range: -90° to +90° (text never appears upside down)

**Map Style Integration:**
The viewer style uses:
- `"text-rotation-alignment": "map"` - Rotates with map, not viewport
- `"text-rotate": ["get", "label_rotation"]` - Uses the calculated rotation angle

In [ ]:
# Generate centroids for administrative boundary labels
print("=== STEP 2b: GENERATING CENTROIDS FOR ADMINISTRATIVE LABELS ===")

# Import the centroid generation function
from scripts import batch_generate_centroids

# Boundary polygon layers that need interior centroids for label placement.
# Centroids are written alongside their source polygons in SCRATCH_BOUNDARIES_DIR.
layers_for_centroids = [
    # COD
    'GRID3_COD_provinces_v8_0',
    'GRID3_COD_antenne_v8_0',
    'GRID3_COD_health_zones_v8_0',
    'GRID3_COD_health_areas_v8_0',
    # NGA
    'GRID3_NGA_operational_states_v2_0',
    'GRID3_NGA_operational_LGAs_v2_0',
    'GRID3_NGA_operational_wards_v2_0',
]

centroid_results = batch_generate_centroids(
    input_dir=str(SCRATCH_BOUNDARIES_DIR),
    output_dir=str(SCRATCH_BOUNDARIES_DIR),
    layers=layers_for_centroids,
    suffix='_centroids',
    verbose=CONFIG["download"]["verbose"]
)

print(f"\nCentroid Generation Summary:")
print(f"  Total layers: {centroid_results['total_layers']}")
print(f"  Successful: {centroid_results['successful']}")
print(f"  Failed: {centroid_results['failed']}")
print(f"  Note: Each centroid includes 'label_rotation' for best-fit orientation")

for layer in centroid_results['layers']:
    if layer['success']:
        output_name = Path(layer['output_file']).name
        print(f"  ✓ {output_name}: {layer['feature_count']:,} centroids")
    else:
        print(f"  ✗ {Path(layer['input_file']).name}: {layer.get('error', 'Unknown error')}")

# List centroid files now in SCRATCH_BOUNDARIES_DIR
if SCRATCH_BOUNDARIES_DIR.exists():
    centroid_files = sorted(SCRATCH_BOUNDARIES_DIR.glob("*_centroids.fgb"))
    print(f"\n{'='*50}")
    print(f"Centroid files in GRID3_boundaries/: {len(centroid_files)}")
    for f in centroid_files:
        print(f"  - {f.name}")

### progress/sanity check

In [ ]:
# Check what files were created during download / conversion
print("=== CHECKING DOWNLOADED FILES ===")

# Check Overture Maps downloads (GeoParquet)
overture_files = []
for data_dir in [CONFIG["paths"]["data_dir"], CONFIG["paths"]["overture_data_dir"]]:
    if data_dir.exists():
        for pattern in CONFIG["download"]["output_formats"]:
            overture_files.extend(data_dir.glob(pattern))

print(f"\nOverture Maps (GeoParquet): {len(overture_files)} files")
overture_size_mb = sum(f.stat().st_size / 1024 / 1024 for f in overture_files)
for f in sorted(overture_files):
    print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

# Check FlatGeobuf files in each thematic scratch subdir
print(f"\nFlatGeobuf (Scratch subdirs):")
fgb_total, fgb_size_mb = 0, 0.0
for label, subdir in [
    ("GRID3_boundaries",       SCRATCH_BOUNDARIES_DIR),
    ("GRID3_settlementExtents", SCRATCH_EXTENTS_DIR),
    ("GRID3_POIs",              SCRATCH_POIS_DIR),
]:
    if subdir.exists():
        files = sorted(subdir.glob("*.fgb"))
        size_mb = sum(f.stat().st_size / 1024 / 1024 for f in files)
        fgb_total += len(files)
        fgb_size_mb += size_mb
        print(f"  {label}/  ({len(files)} files, {size_mb:.1f} MB)")
        for f in files:
            print(f"    {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  {label}/  (directory not yet created)")

# Check PMTiles output
pmtiles_files = sorted(CONFIG["paths"]["tile_dir"].glob("*.pmtiles")) if CONFIG["paths"]["tile_dir"].exists() else []
pmtiles_size_mb = sum(f.stat().st_size / 1024 / 1024 for f in pmtiles_files)
print(f"\nPMTiles (Output): {len(pmtiles_files)} files")
for f in pmtiles_files:
    print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

print(f"\n{'='*50}")
print(f"DISK USAGE SUMMARY")
print(f"{'='*50}")
print(f"  Overture GeoParquet: {overture_size_mb:.1f} MB ({len(overture_files)} files)")
print(f"  FlatGeobuf (GRID3):  {fgb_size_mb:.1f} MB ({fgb_total} files)")
print(f"  PMTiles:             {pmtiles_size_mb:.1f} MB ({len(pmtiles_files)} files)")
print(f"  Total:               {overture_size_mb + fgb_size_mb + pmtiles_size_mb:.1f} MB")

## Convert GeoParquet to FlatGeobuf

Convert downloaded Overture GeoParquet files to FlatGeobuf format for tippecanoe compatibility

**Note**: ArcGIS data was already downloaded as FlatGeobuf in Step 2a, so this step only processes Overture Maps data. Both sources will coexist in the scratch directory.

### Disk Space Management

**Automatic cleanup enabled**: Source `.parquet` files are automatically removed after successful conversion to save disk space.

**Why this works:**
- **FlatGeobuf files (.fgb)** are retained as intermediary files
- PMTiles can be quickly regenerated from .fgb files anytime
- .fgb files are ~30-50% smaller than .parquet
- Conversion from .fgb -> .pmtiles is faster than .parquet -> .pmtiles


**To disable autoremove cleanup:** Set `cleanup_source=False` in the conversion step below.

In [ ]:
# Convert GeoParquet files to FlatGeobuf for optimal tiling performance
print("\n" + "="*70)
print("🔄 STEP 3: CONVERTING OVERTURE GEOPARQUET TO FLATGEOBUF")
print("="*70)
print("\nℹ️  Note: ArcGIS/local GRID3 data already in FlatGeobuf format (from Steps 2a/3a)")
print(f"🧹 Cleanup: {'ENABLED' if CONFIG['fgb_conversion']['cleanup_source'] else 'DISABLED'}")

# Snapshot GRID3 FGB files (in thematic subdirs) before Overture conversion
grid3_fgb_stems = set()
for subdir in [SCRATCH_BOUNDARIES_DIR, SCRATCH_EXTENTS_DIR, SCRATCH_POIS_DIR]:
    if subdir.exists():
        grid3_fgb_stems.update(f.stem for f in subdir.glob("*.fgb"))

if grid3_fgb_stems:
    print(f"\n✓ Preserving {len(grid3_fgb_stems)} existing GRID3 FlatGeobuf file(s) in thematic subdirs")

# Convert Overture GeoParquet files to FlatGeobuf (written to scratch root)
print("\n" + "="*70)

fgb_results = batch_convert_directory(
    input_dir=str(CONFIG["paths"]["overture_data_dir"]),
    output_dir=str(CONFIG["paths"]["scratch_dir"]),
    pattern=CONFIG["fgb_conversion"]["input_pattern"],
    overwrite=CONFIG["fgb_conversion"]["overwrite"],
    verbose=CONFIG["fgb_conversion"]["verbose"],
    cleanup_source=CONFIG["fgb_conversion"]["cleanup_source"]
)

# Summary
overture_fgb_files = list(CONFIG["paths"]["scratch_dir"].glob("*.fgb")) if CONFIG["paths"]["scratch_dir"].exists() else []

print("="*70)
print("FLATGEOBUF FILES READY FOR TILING")
print("="*70)
print(f"  Overture FGBs (scratch root): {len(overture_fgb_files)}")
print(f"  GRID3 FGBs (thematic subdirs): {len(grid3_fgb_stems)}")

if CONFIG["fgb_conversion"]["cleanup_source"]:
    print(f"\n  ℹ  Source .parquet files removed to save disk space")

if overture_fgb_files:
    print(f"\n  Overture files (Protomaps handles tiling, not tippecanoe):")
    for f in sorted(overture_fgb_files)[:10]:
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"     • {f.name} ({size_mb:.1f} MB)")
    if len(overture_fgb_files) > 10:
        print(f"     ... and {len(overture_fgb_files) - 10} more")

print("="*70 + "\n")

## Convert Local Geospatial Data to FlatGeobuf 

For local data sources (feature classes, shapefiles, geopackages, etc.) that don't need API download

In [ ]:
# OPTIONAL: Convert local geospatial data to FlatGeobuf
# Uncomment and customize the examples below for your data sources

print("=== STEP 3a: CONVERTING LOCAL GEOSPATIAL DATA (OPTIONAL) ===\n")

# ============================================================
# EXAMPLE 1: Convert Shapefile  →  boundaries subdir
# ============================================================
# convert_geodata_to_fgb(
#     input_path="/path/to/shapefile.shp",
#     output_path=str(SCRATCH_BOUNDARIES_DIR / "my_layer.fgb"),
#     clip_extent=CONFIG["extent"]["coordinates"],
#     overwrite=True,
#     verbose=True
# )

# ============================================================
# EXAMPLE 2: Convert ArcGIS File Geodatabase  →  extents subdir
# ============================================================
convert_geodata_to_fgb(
    input_path="/mnt/d/mheaton/grid3_tiles/data/1-input/grid3/GRID3_NGA_settlement_extents_v3_1.gdb",
    output_path=str(SCRATCH_EXTENTS_DIR / "GRID3_NGA_settlement_extents_v3_1"),
    layer="GRID3_NGA_settlement_extents_v3_1",  # Required for multi-layer sources e.g. gdb
    clip_extent=None,
    overwrite=True,
    verbose=True
)

# ============================================================
# EXAMPLE 3: Convert GeoPackage  →  POI subdir
# ============================================================
# convert_geodata_to_fgb(
#     input_path="/path/to/data.gpkg",
#     output_path=str(SCRATCH_POIS_DIR / "settlement_names.fgb"),
#     layer="settlement_names",
#     where="population > 100000",
#     overwrite=True,
#     verbose=True
# )

# ============================================================
# EXAMPLE 4: Batch convert multiple files
# ============================================================
# batch_results = batch_convert_geodata(
#     input_paths=[Path("/path/to/file1.shp"), Path("/path/to/file2.geojson")],
#     output_dir=str(SCRATCH_BOUNDARIES_DIR),  # or SCRATCH_EXTENTS_DIR / SCRATCH_POIS_DIR
#     clip_extent=CONFIG["extent"]["coordinates"],
#     overwrite=True,
#     verbose=True
# )

print("✓ Step 3a complete - Edit cell above to convert your local data\n")
print("  Converted files are saved to thematic scratch subdirectories:")
print(f"  Boundaries:         {SCRATCH_BOUNDARIES_DIR}")
print(f"  Settlement extents: {SCRATCH_EXTENTS_DIR}")
print(f"  POIs:               {SCRATCH_POIS_DIR}")

## Process FlatGeobuf to PMTiles

Use the `runCreateTiles.py` module to convert FlatGeobuf files to PMTiles using custom tippecanoe queries from tippecanoe.py

In [2]:
# Step 4: Process all geospatial files to PMTiles
print("=== STEP 4: PROCESSING TO PMTILES ===")

# Process all downloaded and converted files to PMTiles using CONFIG settings
# Now supports: GeoJSON, GeoJSONSeq, and GeoParquet formats
tiling_results = process_to_tiles(
    extent=None,
    # extent=CONFIG["extent"]["coordinates"],
    input_dirs=[str(d) for d in CONFIG["tiling"]["input_dirs"]],  # Convert Path objects to strings
    filter_pattern=CONFIG["tiling"]["filter_pattern"],  # Pass filter pattern from CONFIG
    output_dir=str(CONFIG["tiling"]["output_dir"]),  # Use explicit output directory from CONFIG
    parallel=CONFIG["tiling"]["parallel"],
    verbose=CONFIG["tiling"]["verbose"]
)

# print(f"Tiling completed: {tiling_results['success']}")
# print(f"Files processed: {len(tiling_results['processed_files'])}/{tiling_results['total_files']}")

if tiling_results["errors"]:
    print(f"Errors encountered: {len(tiling_results['errors'])}")
    for error in tiling_results["errors"]:
        print(f"  - {error}")

# Display generated PMTiles files
if tiling_results["processed_files"]:
    print(f"\n✓ Successfully generated {len(tiling_results['processed_files'])} PMTiles:")
    
    pmtiles_files = list(CONFIG["paths"]["tile_dir"].glob("*.pmtiles"))
    
    total_size_mb = 0
    for pmtile in sorted(pmtiles_files):
        size_mb = pmtile.stat().st_size / 1024 / 1024
        total_size_mb += size_mb
        print(f"  {pmtile.name} ({size_mb:.1f} MB)")
    
    print(f"\nTotal PMTiles size: {total_size_mb:.1f} MB")
    print(f"Files location: {CONFIG['paths']['tile_dir']}")
    
else:
    print("\nNo PMTiles files were generated. Check the errors above.")
    print(f"Make sure you have geospatial files (GeoJSON/GeoJSONSeq/GeoParquet) in: {[str(d) for d in CONFIG['tiling']['input_dirs']]}")

=== STEP 4: PROCESSING TO PMTILES ===
=== PROCESSING TO TILES ===
Found 3 files to process:
  GRID3_COD_settlement_extents_v3_1.fgb (FlatGeobuf) [group]
  GRID3_NGA_settlement_extents_v3_0.fgb (FlatGeobuf) [group]
  GRID3_NGA_settlement_extents_v4_0.fgb (FlatGeobuf) [group]
  → group 'GRID3_COD_settlement_extents': GRID3_COD_settlement_extents.pmtiles
  → group 'GRID3_NGA_settlement_extents': GRID3_NGA_settlement_extents.pmtiles
  → group 'GRID3_NGA_POIs': GRID3_NGA_POIs.pmtiles

Processing group 'GRID3_COD_settlement_extents'...


detected indexed FlatGeobuf: assigning feature IDs by sequence
572537 features, 286155745 bytes of geometry and attributes, 53733790 bytes of string pool, 2687935680 bytes of vertices, 13972368 bytes of nodes
Choosing a maxzoom of -z7 for features typically 2617 feet (798 meters) apart, and at least 678 feet (207 meters) apart
Choosing a maxzoom of -z15 for resolution of about 9 feet (2 meters) within features
Using base zoom of -z15
  99.9%  15/19181/16170  
  100.0%  15/18318/15989  

✓ GRID3_COD_settlement_extents → /tmp/grid3_tiles/data/3-pmtiles/GRID3_COD_settlement_extents.pmtiles

Processing group 'GRID3_NGA_settlement_extents'...


detected indexed FlatGeobuf: assigning feature IDs by sequence
detected indexed FlatGeobuf: assigning feature IDs by sequence
3304056 features, 1504675886 bytes of geometry and attributes, 864730360 bytes of string pool, 9697466736 bytes of vertices, 73405072 bytes of nodes
Choosing a maxzoom of -z9 for features typically 671 feet (205 meters) apart, and at least 128 feet (39 meters) apart
Choosing a maxzoom of -z15 for resolution of about 14 feet (4 meters) within features
Using base zoom of -z15
tile 7/68/59 size is 10581999 with detail 12, >5120000    
Going to try keeping the biggest 36.29% of the features to make it fit
tile 7/65/59 size is 17609149 with detail 12, >5120000    
Going to try keeping the biggest 21.81% of the features to make it fit
tile 7/68/59 size is 8138699 with detail 12, >5120000    
Going to try keeping the biggest 17.12% of the features to make it fit
tile 7/65/60 size is 18576597 with detail 12, >5120000    
Going to try keeping the biggest 20.67% of the fe

✓ GRID3_NGA_settlement_extents → /tmp/grid3_tiles/data/3-pmtiles/GRID3_NGA_settlement_extents.pmtiles

Processing group 'GRID3_NGA_POIs'...
✓ GRID3_NGA_POIs → /tmp/grid3_tiles/data/3-pmtiles/GRID3_NGA_POIs.pmtiles

=== TILE PROCESSING COMPLETE ===
Processed: 3/3 files

✓ Successfully generated 3 PMTiles:
  GRID3_COD_POIs.pmtiles (18.0 MB)
  GRID3_COD_boundaries.pmtiles (98.8 MB)
  GRID3_COD_settlement_extents.pmtiles (640.6 MB)
  GRID3_NGA_boundaries.pmtiles (41.7 MB)
  GRID3_NGA_settlement_extents.pmtiles (4581.1 MB)
  base.pmtiles (31810.8 MB)
  places.pmtiles (2761.0 MB)
  protomaps_base.pmtiles (129751.2 MB)

Total PMTiles size: 169703.3 MB
Files location: /tmp/grid3_tiles/data/3-pmtiles
